# Post-OFAT DL All Models + Tuning: OLLAMA

Ноутбук читает готовые OFAT feature-артефакты, оценивает модели и запускает тюнинг для лучших схем.


In [ ]:
import json
import math
import os
import re
import time
import warnings
from pathlib import Path
from datetime import datetime

_mpl_cache_dir = Path(os.getenv("MPLCONFIGDIR", "/tmp/matplotlib-cache"))
_mpl_cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(_mpl_cache_dir)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


In [ ]:
SEED = 2
DURATION_CAP = 960
TARGET_COL = "duration_hours"
PROVIDER = "ollama"
RUN_LABEL = "post_ofat_all_dl_models_with_tuning"

# Set these to True if you intentionally want to overwrite existing outputs.
FORCE_RERUN_BASELINE = False
FORCE_RERUN_TUNING = False

def require_path(env_name: str, label: str) -> Path:
    raw_value = os.environ.get(env_name, "").strip()
    if not raw_value:
        raise FileNotFoundError(f"Укажи путь к {label} через переменную окружения {env_name}.")
    path = Path(raw_value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


def optional_path(env_name: str, default_value: str | Path) -> Path:
    return Path(os.environ.get(env_name, default_value)).expanduser()

PROJECT_DIR = optional_path("OLLAMA_OFAT_PROJECT_DIR", "./ollama_meta_prompt_ofat_fixed_features")
INPUT_DIR = optional_path("LLM_INPUT_DIR", PROJECT_DIR / "inputs")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
MODEL_TAG = re.sub(r"[^a-zA-Z0-9._-]+", "_", OLLAMA_MODEL).replace(":", "_")
DATA_PATH = require_path("DATA_FINALL_PATH", "data_finall")
ARTIFACT_ROOT = optional_path("OLLAMA_OFAT_ARTIFACT_ROOT", "./ollama_meta_prompt_ofat_fixed_features/artifacts_fixed_features_v7_llama3.2_3b")
OUTPUT_DIR = optional_path("POST_OFAT_DL_OLLAMA_OUTPUT_DIR", "./artifacts_post_ofat_dl_all_models_with_tuning_ollama")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [DATA_PATH]
ARTIFACT_ROOT_CANDIDATES = [ARTIFACT_ROOT]

print("Provider:", PROVIDER)
print("Output dir:", OUTPUT_DIR.resolve())


In [ ]:
def resolve_existing_path(candidates, label):
    for p in candidates:
        p = Path(p)
        if p.exists():
            print(f"{label}: {p.resolve()}")
            return p
    raise FileNotFoundError(f"{label} not found. Tried: {[str(Path(p)) for p in candidates]}")


DATA_PATH = resolve_existing_path(DATA_CANDIDATES, "Data")
ARTIFACT_ROOT = resolve_existing_path(ARTIFACT_ROOT_CANDIDATES, "OFAT artifact root")


df = pd.read_csv(DATA_PATH).copy()
df["source_row_id"] = np.arange(len(df), dtype=int)

split = int(len(df) * 0.8)
train_full = df.iloc[:split].copy()
test_full = df.iloc[split:].copy().reset_index(drop=True)
train_filt = train_full[train_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)
val_start = int(len(train_filt) * 0.75)
train_core = train_filt.iloc[:val_start].copy().reset_index(drop=True)
val_fresh = train_filt.iloc[val_start:].copy().reset_index(drop=True)
test_typical = test_full[test_full[TARGET_COL] < DURATION_CAP].copy().reset_index(drop=True)

meta_train_core = train_core.drop(columns=[TARGET_COL], errors="ignore").copy()
meta_val_fresh = val_fresh.drop(columns=[TARGET_COL], errors="ignore").copy()
meta_test_full = test_full.drop(columns=[TARGET_COL], errors="ignore").copy()

y_train_core = train_core[TARGET_COL].reset_index(drop=True)
y_val_fresh = val_fresh[TARGET_COL].reset_index(drop=True)
y_train_all = pd.concat([y_train_core, y_val_fresh], ignore_index=True)
y_test_full = test_full[TARGET_COL].reset_index(drop=True)
y_test_typical = test_typical[TARGET_COL].reset_index(drop=True)

print("full df      :", df.shape)
print("train_filt   :", train_filt.shape)
print("train_core   :", train_core.shape)
print("val_fresh    :", val_fresh.shape)
print("test_full    :", test_full.shape)
print("test_typical :", test_typical.shape)


In [ ]:
FEATURE_COLS = [
    "execution_complexity_1_5",
    "coordination_risk_1_5",
    "testing_effort_1_5",
    "uncertainty_1_5",
    "urgent_delivery_0_1",
    "likely_long_task_0_1",
]

STAGE_ORDER = [
    ("stage_1_role", 1),
    ("stage_2_project", 2),
    ("stage_3_goal", 3),
    ("stage_4_granularity", 4),
    ("stage_5_long_task_policy", 5),
]


def reg_metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MdAE": median_absolute_error(y_true, pred),
    }


def read_agg(config_dir, split_name):
    path = Path(config_dir) / f"aggregated_features__{split_name}.csv"
    if not path.exists():
        return None
    return pd.read_csv(path)


def prepare_model_matrix(meta_df, agg_df):
    merged = meta_df.merge(agg_df, on="source_row_id", how="left").copy()
    for c in FEATURE_COLS:
        if c not in merged.columns:
            merged[c] = 0.0
    drop_cols = [
        c
        for c in merged.columns
        if c.endswith("__std")
        or c.endswith("__agree")
        or c.endswith("__p1")
        or c.endswith("__var")
        or c in {"raw_text", "latency_s", "mean_latency_s"}
    ]
    merged = merged.drop(columns=drop_cols, errors="ignore")
    merged = merged.drop(columns=[TARGET_COL], errors="ignore")
    merged = merged.select_dtypes(include=[np.number, "bool"]).copy()
    merged = merged.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return merged


def discover_ofat_configs(artifact_root):
    rows = []
    for stage_name, stage_num in STAGE_ORDER:
        stage_dir = Path(artifact_root) / stage_name
        if not stage_dir.exists():
            print("Missing stage dir:", stage_dir)
            continue
        for config_dir in sorted(p for p in stage_dir.iterdir() if p.is_dir()):
            required = [
                config_dir / "aggregated_features__train_core.csv",
                config_dir / "aggregated_features__val_fresh.csv",
                config_dir / "aggregated_features__test_full.csv",
            ]
            if all(p.exists() for p in required):
                rows.append({
                    "provider": PROVIDER,
                    "stage": stage_name,
                    "stage_num": stage_num,
                    "config_slug": config_dir.name,
                    "config_dir": config_dir,
                })
    out = pd.DataFrame(rows).sort_values(["stage_num", "config_slug"]).reset_index(drop=True)
    if len(out) != 20:
        print(f"WARNING: expected 20 OFAT configs, found {len(out)}")
    return out


def load_config_matrices(config_dir):
    agg_train_core = read_agg(config_dir, "train_core")
    agg_val_fresh = read_agg(config_dir, "val_fresh")
    agg_test_full = read_agg(config_dir, "test_full")
    if agg_train_core is None or agg_val_fresh is None or agg_test_full is None:
        raise FileNotFoundError(f"Config is missing required aggregate files: {config_dir}")

    X_train_core = prepare_model_matrix(meta_train_core, agg_train_core)
    X_val_fresh = prepare_model_matrix(meta_val_fresh, agg_val_fresh)
    X_train_all = pd.concat([X_train_core, X_val_fresh], ignore_index=True)

    X_test_full = prepare_model_matrix(meta_test_full, agg_test_full)
    typical_ids = set(test_typical["source_row_id"].tolist())
    X_test_typical = X_test_full.loc[test_full["source_row_id"].isin(typical_ids)].reset_index(drop=True)
    return X_train_all, X_test_typical, X_test_full



ofat_configs = discover_ofat_configs(ARTIFACT_ROOT)
ofat_configs.to_csv(OUTPUT_DIR / "ofat_configs_discovered.csv", index=False)
display(ofat_configs[["stage", "stage_num", "config_slug", "config_dir"]])


## DL runtime and architectures from DL_baseline / DL_tuning

In [ ]:
import os

TABPFN_ENABLE_HF_LOGIN = False
TABPFN_FORCE_MODEL_VERSION = "V2_5"
TABPFN_HF_TOKEN_EFFECTIVE = os.environ.get("HF_TOKEN", "").strip()
TABPFN_HF_LOGIN_STATUS = "not_used"


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import time, math, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)


SEED = 2
seed = SEED
torch.manual_seed(seed)
np.random.seed(seed)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Device:', device)

def seed_everything(s=SEED):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

INPUT_DIM = 1

# ═══════════════════════════════════════════════════════════════
#  2. АРХИТЕКТУРЫ  (24 шт.)
# ═══════════════════════════════════════════════════════════════

# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  3. ФУНКЦИИ ОБУЧЕНИЯ
# ═══════════════════════════════════════════════════════════════

def train_model(model, epochs=300, lr=1e-3, wd=1e-4, bs=256,
                patience=30, aux_w=0.0, trial=None):
    """Обучение с early stopping. Возвращает (model, val_mae, epochs_used)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
        if trial is not None:
            trial.report(v_mae, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                  step_size=0.1, bs=256, patience=30, dropout=0.1, trial=None):
    """Gradient boosting с NN base learners. Возвращает (stages, val_mae, total_epochs)."""
    stages = []
    X_v, y_v = X_va_t.to(device), y_va_t.to(device)
    ds = TensorDataset(X_tr_t, y_tr_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    # Предсказание ансамбля на train/val
    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    total_ep = 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
            total_ep += 1
        model.load_state_dict(best_stage_state)
        stages.append(model)
        # Проверяем ансамбль
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
        if trial is not None:
            trial.report(ens_val, stage_idx)
            if trial.should_prune():
                raise optuna.TrialPruned()
    stages = stages[:best_n]
    return stages, best_val, total_ep


def eval_on_test(model, name=""):
    """Evaluate model on holdout test."""
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


def eval_grownet_on_test(stages, step_size=0.1):
    """Evaluate GrowNet ensemble on holdout test."""
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    mae = mean_absolute_error(ytest, pred)
    rmse = np.sqrt(mean_squared_error(ytest, pred))
    mdae = median_absolute_error(ytest, pred)
    return mae, rmse, mdae


print("24 архитектуры + train_model + train_grownet определены.")


In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# ═══════════════════════════════════════════════════════════════
#  Baseline: все 24 архитектуры с дефолтными параметрами
# ═══════════════════════════════════════════════════════════════

DEFAULTS = {
    # MLP-семейство
    "MLP":            lambda: MLP(INPUT_DIM, [256, 128], dropout=0.3),
    "ResMLP":         lambda: ResMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "SNN":            lambda: SNN(INPUT_DIM, hidden_dims=[256, 128], alpha_dropout=0.1),
    "GatedMLP":       lambda: GatedMLP(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "GANDALF":        lambda: GANDALF(INPUT_DIM, hidden=256, n_blocks=3, dropout=0.3),
    "DAE-MLP":        lambda: DAEMLP(INPUT_DIM, hidden=256, latent=64, noise=0.3, dropout=0.3),
    # CNN / RNN
    "CNN1D":          lambda: CNN1D(INPUT_DIM, channels=[64, 128, 64], ks=5, dropout=0.3),
    "BiGRU":          lambda: BiGRU(INPUT_DIM, hidden=64, n_layers=2, dropout=0.3),
    # Transformer / Attention
    "FT-Transformer": lambda: FTTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "TabTransformer": lambda: TabTransformer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2),
    "SAINT":          lambda: SAINT(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "AutoInt":        lambda: AutoInt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "Trompt":         lambda: Trompt(INPUT_DIM, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
    "ExcelFormer":    lambda: ExcelFormer(INPUT_DIM, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
    "ARM-Net":        lambda: ARMNet(INPUT_DIM, n_exp=64, hidden=128, order=2, dropout=0.2),
    # Tree-inspired
    "NODE":           lambda: NODE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "GRANDE":         lambda: GRANDE(INPUT_DIM, n_trees=32, depth=4, dropout=0.0),
    "Net-DNF":        lambda: NetDNF(INPUT_DIM, n_formulas=64, n_conjuncts=6, dropout=0.2),
    "TabNet":         lambda: TabNet(INPUT_DIM, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1),
    # Retrieval / Special
    "TabR":           lambda: TabR(INPUT_DIM, hidden=256, n_layers=3, k=32, dropout=0.3),
    "SwitchTab":      lambda: SwitchTab(INPUT_DIM, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3),
    "PTaRL":          lambda: PTaRL(INPUT_DIM, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3),
    "DCNv2":          lambda: DCNv2(INPUT_DIM, n_cross=3, hidden=256, dropout=0.3),
}

AUX_W = {"DAE-MLP": 0.1, "TabNet": 0.01, "SwitchTab": 0.1}
def build_arch_from_params(arch_name: str, bp: dict):
    if arch_name == "MLP":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return MLP(INPUT_DIM, dims, bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ResMLP":
        return ResMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SNN":
        dims = [max(int(bp["first_hidden"]) // (2 ** i), 32) for i in range(int(bp["n_layers"]))]
        return SNN(INPUT_DIM, dims, bp["alpha_dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GatedMLP":
        return GatedMLP(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GANDALF":
        return GANDALF(INPUT_DIM, bp["hidden"], bp["n_blocks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DAE-MLP":
        return DAEMLP(INPUT_DIM, bp["hidden"], bp["latent"], bp["noise"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "CNN1D":
        nc = int(bp["n_conv"]); bc = int(bp["base_ch"])
        chs = [bc * (2 ** i) for i in range(nc // 2 + 1)]
        chs += [bc * (2 ** i) for i in range(nc // 2 - 1, -1, -1)]
        chs = chs[:nc]
        return CNN1D(INPUT_DIM, chs, bp["ks"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "BiGRU":
        return BiGRU(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "FT-Transformer":
        return FTTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabTransformer":
        return TabTransformer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["mlp_hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SAINT":
        return SAINT(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "AutoInt":
        return AutoInt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Trompt":
        return Trompt(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ExcelFormer":
        return ExcelFormer(INPUT_DIM, bp["d_model"], bp["n_heads"], bp["n_layers"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "ARM-Net":
        return ARMNet(INPUT_DIM, bp["n_exp"], bp["hidden"], bp["order"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "NODE":
        return NODE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "GRANDE":
        return GRANDE(INPUT_DIM, bp["n_trees"], bp["depth"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "Net-DNF":
        return NetDNF(INPUT_DIM, bp["n_formulas"], bp["n_conjuncts"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabNet":
        return TabNet(INPUT_DIM, bp["n_steps"], bp["n_d"], bp["n_a"], bp["relax"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "TabR":
        return TabR(INPUT_DIM, bp["hidden"], bp["n_layers"], bp["k"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "SwitchTab":
        return SwitchTab(INPUT_DIM, bp["d_enc"], bp["d_mutual"], bp["d_salient"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "PTaRL":
        return PTaRL(INPUT_DIM, bp["n_prototypes"], bp["d_latent"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    elif arch_name == "DCNv2":
        return DCNv2(INPUT_DIM, bp["n_cross"], bp["hidden"], bp["dropout"]), bp.get("aux_w", 0.0)
    else:
        raise KeyError(f"Unknown architecture: {arch_name}")

def serialize_params(params: dict):
    clean = {}
    for k, v in params.items():
        if isinstance(v, (np.integer,)):
            clean[k] = int(v)
        elif isinstance(v, (np.floating,)):
            clean[k] = float(v)
        else:
            clean[k] = v
    return clean

In [ ]:
torch.manual_seed(seed)
np.random.seed(seed)

# ═══════════════════════════════════════════════════════════════
#  Optuna objectives + baseline-aware honest tuning
# ═══════════════════════════════════════════════════════════════

BASELINE_TRIAL_PARAMS = {
    "MLP":            {"n_layers": 2, "first_hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ResMLP":         {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SNN":            {"n_layers": 2, "first_hidden": 256, "alpha_dropout": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GatedMLP":       {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GANDALF":        {"hidden": 256, "n_blocks": 3, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "DAE-MLP":        {"hidden": 256, "latent": 64, "noise": 0.3, "dropout": 0.3, "aux_w": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "CNN1D":          {"n_conv": 3, "base_ch": 64, "ks": 5, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "BiGRU":          {"hidden": 64, "n_layers": 2, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "FT-Transformer": {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabTransformer": {"d_model": 32, "n_heads": 4, "n_layers": 2, "mlp_hidden": 128, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SAINT":          {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "AutoInt":        {"d_model": 32, "n_heads": 4, "n_layers": 3, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "Trompt":         {"d_model": 32, "n_heads": 4, "n_layers": 3, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ExcelFormer":    {"d_model": 32, "n_heads": 4, "n_layers": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "ARM-Net":        {"n_exp": 64, "hidden": 128, "order": 2, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "NODE":           {"n_trees": 32, "depth": 4, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GRANDE":         {"n_trees": 32, "depth": 4, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "Net-DNF":        {"n_formulas": 64, "n_conjuncts": 6, "dropout": 0.2, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabNet":         {"n_steps": 3, "n_d": 64, "n_a": 64, "relax": 1.5, "dropout": 0.1, "aux_w": 0.01, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "TabR":           {"hidden": 256, "n_layers": 3, "k": 32, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "SwitchTab":      {"d_enc": 128, "d_mutual": 64, "d_salient": 64, "dropout": 0.3, "aux_w": 0.1, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "PTaRL":          {"n_prototypes": 16, "d_latent": 128, "hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "DCNv2":          {"n_cross": 3, "hidden": 256, "dropout": 0.3, "lr": 1e-3, "wd": 1e-4, "bs": 256},
    "GrowNet":        {"n_stages": 5, "hidden": 32, "step_size": 0.1, "dropout": 0.1, "lr": 0.01, "wd": 1e-4, "bs": 256},
}

def _seed_trial():
    # Один и тот же seed для каждого trial -> сравнение гиперпараметров честнее и ближе к baseline.
    seed_everything(SEED)

def _suggest_opt(trial, *, bs_choices, lr_low=1e-4, lr_high=3e-3, wd_low=1e-5, wd_high=1e-3):
    lr = trial.suggest_float("lr", lr_low, lr_high, log=True)
    wd = trial.suggest_float("wd", wd_low, wd_high, log=True)
    bs = trial.suggest_categorical("bs", bs_choices)
    return lr, wd, bs

def obj_mlp(trial):
    _seed_trial()
    n_layers = trial.suggest_int("n_layers", 2, 4)
    first = trial.suggest_categorical("first_hidden", [128, 256, 512])
    dims = [max(first // (2**i), 32) for i in range(n_layers)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(MLP(INPUT_DIM, dims, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_resmlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(ResMLP(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_snn(trial):
    _seed_trial()
    n_layers = trial.suggest_int("n_layers", 2, 4)
    first = trial.suggest_categorical("first_hidden", [128, 256, 512])
    dims = [max(first // (2**i), 32) for i in range(n_layers)]
    alpha_do = trial.suggest_float("alpha_dropout", 0.01, 0.2)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(SNN(INPUT_DIM, dims, alpha_do), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_gatedmlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(GatedMLP(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_gandalf(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_blocks = trial.suggest_int("n_blocks", 2, 5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(GANDALF(INPUT_DIM, hidden, n_blocks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_daemlp(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 512])
    latent = trial.suggest_categorical("latent", [32, 64, 128])
    noise = trial.suggest_float("noise", 0.1, 0.5)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    aux_w = trial.suggest_float("aux_w", 0.01, 0.5, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(DAEMLP(INPUT_DIM, hidden, latent, noise, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_cnn1d(trial):
    _seed_trial()
    n_conv = trial.suggest_int("n_conv", 2, 4)
    base_ch = trial.suggest_categorical("base_ch", [32, 64, 128])
    chs = [base_ch * (2**i) for i in range(n_conv // 2 + 1)]
    chs += [base_ch * (2**i) for i in range(n_conv // 2 - 1, -1, -1)]
    chs = chs[:n_conv]
    ks = trial.suggest_categorical("ks", [3, 5, 7, 9])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(CNN1D(INPUT_DIM, chs, ks, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_bigru(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [32, 64, 128])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256], wd_low=1e-5)
    return train_model(BiGRU(INPUT_DIM, hidden, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_ft_transformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32, 48])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(FTTransformer(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_tabtransformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    mlp_hidden = trial.suggest_categorical("mlp_hidden", [64, 128, 256])
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(TabTransformer(INPUT_DIM, d_model, n_heads, n_layers, mlp_hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_saint(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(SAINT(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_autoint(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [8, 16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(AutoInt(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_trompt(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 2, 4)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(Trompt(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_excelformer(trial):
    _seed_trial()
    d_model = trial.suggest_categorical("d_model", [16, 32])
    n_heads = trial.suggest_categorical("n_heads", [2, 4])
    n_layers = trial.suggest_int("n_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.05, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[64, 128, 256], lr_low=5e-5, lr_high=1e-3, wd_low=1e-6)
    return train_model(ExcelFormer(INPUT_DIM, d_model, n_heads, n_layers, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_armnet(trial):
    _seed_trial()
    n_exp = trial.suggest_categorical("n_exp", [32, 64, 128])
    hidden = trial.suggest_categorical("hidden", [64, 128, 256])
    order = trial.suggest_int("order", 2, 3)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(ARMNet(INPUT_DIM, n_exp, hidden, order, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_node(trial):
    _seed_trial()
    n_trees = trial.suggest_categorical("n_trees", [16, 32, 64, 128])
    depth = trial.suggest_int("depth", 3, 6)
    dropout = trial.suggest_float("dropout", 0.0, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(NODE(INPUT_DIM, n_trees, depth, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_grande(trial):
    _seed_trial()
    n_trees = trial.suggest_categorical("n_trees", [16, 32, 64])
    depth = trial.suggest_int("depth", 3, 5)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512], wd_low=1e-6)
    return train_model(GRANDE(INPUT_DIM, n_trees, depth, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_netdnf(trial):
    _seed_trial()
    n_formulas = trial.suggest_categorical("n_formulas", [32, 64, 128])
    n_conjuncts = trial.suggest_int("n_conjuncts", 4, 8)
    dropout = trial.suggest_float("dropout", 0.1, 0.3)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(NetDNF(INPUT_DIM, n_formulas, n_conjuncts, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_tabnet(trial):
    _seed_trial()
    n_steps = trial.suggest_int("n_steps", 2, 5)
    n_d = trial.suggest_categorical("n_d", [32, 64, 128])
    n_a = trial.suggest_categorical("n_a", [32, 64, 128])
    relax = trial.suggest_float("relax", 1.0, 2.0)
    dropout = trial.suggest_float("dropout", 0.01, 0.2)
    aux_w = trial.suggest_float("aux_w", 0.001, 0.1, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(TabNet(INPUT_DIM, n_steps, n_d, n_a, relax, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_tabr(trial):
    _seed_trial()
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    n_layers = trial.suggest_int("n_layers", 2, 4)
    k = trial.suggest_categorical("k", [16, 32, 64])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256])
    return train_model(TabR(INPUT_DIM, hidden, n_layers, k, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_switchtab(trial):
    _seed_trial()
    d_enc = trial.suggest_categorical("d_enc", [64, 128, 256])
    d_mutual = trial.suggest_categorical("d_mutual", [32, 64, 128])
    d_salient = trial.suggest_categorical("d_salient", [32, 64, 128])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    aux_w = trial.suggest_float("aux_w", 0.01, 0.5, log=True)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(SwitchTab(INPUT_DIM, d_enc, d_mutual, d_salient, dropout), lr=lr, wd=wd, bs=bs, aux_w=aux_w, trial=trial)[1]

def obj_ptarl(trial):
    _seed_trial()
    n_proto = trial.suggest_categorical("n_prototypes", [8, 16, 32])
    d_latent = trial.suggest_categorical("d_latent", [64, 128, 256])
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(PTaRL(INPUT_DIM, n_proto, d_latent, hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_dcnv2(trial):
    _seed_trial()
    n_cross = trial.suggest_int("n_cross", 2, 5)
    hidden = trial.suggest_categorical("hidden", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    lr, wd, bs = _suggest_opt(trial, bs_choices=[128, 256, 512])
    return train_model(DCNv2(INPUT_DIM, n_cross, hidden, dropout), lr=lr, wd=wd, bs=bs, trial=trial)[1]

def obj_grownet(trial):
    _seed_trial()
    n_stages = trial.suggest_int("n_stages", 3, 8)
    hidden = trial.suggest_categorical("hidden", [16, 32, 64])
    step_size = trial.suggest_float("step_size", 0.05, 0.3)
    dropout = trial.suggest_float("dropout", 0.0, 0.2)
    lr = trial.suggest_float("lr", 1e-3, 0.05, log=True)
    wd = trial.suggest_float("wd", 1e-5, 1e-3, log=True)
    bs = trial.suggest_categorical("bs", [128, 256, 512])
    _, val_mae, _ = train_grownet(n_stages, hidden, lr, wd, step_size, bs, patience=30, dropout=dropout, trial=trial)
    return val_mae


In [ ]:
def clear_device_cache():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()

def calc_reg_metrics(y_true, pred):
    return {
        "MAE": float(mean_absolute_error(y_true, pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, pred))),
        "MdAE": float(median_absolute_error(y_true, pred)),
    }

def eval_on_typical_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_on_full_holdout(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def eval_grownet_on_typical_holdout(stages, step_size=0.1):
    X_t = X_te_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_grownet_on_full_holdout(stages, step_size=0.1):
    X_t = X_te_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_model_fulltrain(model, epochs=100, lr=1e-3, wd=1e-4, bs=256, aux_w=0.0):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1), eta_min=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad()
            loss.backward()
            opt.step()
        sched.step()
    return model

def eval_fulltrain_model_typical(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_full_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_model_full(model):
    model.eval()
    with torch.no_grad():
        pred = model(X_te_fullscale_stress_t.to(device)).cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

def train_grownet_fulltrain(n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                            step_size=0.1, bs=256, epochs_per_stage=50, dropout=0.1):
    stages = []
    ds = TensorDataset(X_full_t, y_full_t)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    for stage_idx in range(n_stages):
        model = GrowNetStage(INPUT_DIM, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_per_stage, 1), eta_min=1e-6)

        for ep in range(epochs_per_stage):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()

        stages.append(model)
    return stages

def eval_fulltrain_grownet_typical(stages, step_size=0.1):
    X_t = X_te_full_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_typical, pred)

def eval_fulltrain_grownet_full(stages, step_size=0.1):
    X_t = X_te_fullscale_stress_t.to(device)
    p = torch.zeros(X_t.size(0), 1, device=device)
    for s in stages:
        s.eval()
        with torch.no_grad():
            p = p + step_size * s(X_t, p)
    pred = p.cpu().numpy().flatten()
    return calc_reg_metrics(ytest_full, pred)

print("Full-holdout helpers defined.")

In [ ]:

optuna.logging.set_verbosity(optuna.logging.WARNING)

OBJECTIVE_MAP = {
    "MLP": obj_mlp,
    "ResMLP": obj_resmlp,
    "SNN": obj_snn,
    "GatedMLP": obj_gatedmlp,
    "GANDALF": obj_gandalf,
    "DAE-MLP": obj_daemlp,
    "CNN1D": obj_cnn1d,
    "BiGRU": obj_bigru,
    "FT-Transformer": obj_ft_transformer,
    "TabTransformer": obj_tabtransformer,
    "SAINT": obj_saint,
    "AutoInt": obj_autoint,
    "Trompt": obj_trompt,
    "ExcelFormer": obj_excelformer,
    "ARM-Net": obj_armnet,
    "NODE": obj_node,
    "GRANDE": obj_grande,
    "Net-DNF": obj_netdnf,
    "TabNet": obj_tabnet,
    "TabR": obj_tabr,
    "SwitchTab": obj_switchtab,
    "PTaRL": obj_ptarl,
    "DCNv2": obj_dcnv2,
    "GrowNet": obj_grownet,
}
def fit_best_inner_model(arch_name: str, bp: dict):
    seed_everything(SEED)
    if arch_name == "GrowNet":
        stages, val_mae, epochs_used = train_grownet(
            n_stages=bp["n_stages"],
            hidden=bp["hidden"],
            lr=bp["lr"],
            wd=bp["wd"],
            step_size=bp["step_size"],
            bs=bp["bs"],
            patience=30,
            dropout=bp["dropout"],
        )
        metrics_typ = eval_grownet_on_typical_holdout(stages, bp["step_size"])
        metrics_full = eval_grownet_on_full_holdout(stages, bp["step_size"])
        n_params = int(sum(sum(p.numel() for p in s.parameters()) for s in stages))
        return {
            "kind": "grownet",
            "fitted": stages,
            "val_mae": float(val_mae),
            "epochs_used": int(epochs_used),
            "n_params": n_params,
            "metrics_typical": metrics_typ,
            "metrics_full": metrics_full,
        }

    model, aux_w = build_arch_from_params(arch_name, bp)
    model, val_mae, epochs_used = train_model(
        model,
        epochs=300,
        lr=bp["lr"],
        wd=bp["wd"],
        bs=bp["bs"],
        patience=30,
        aux_w=aux_w,
    )
    if arch_name == "TabR":
        model.build_store(X_tr_t.to(device), y_tr_t.to(device))
    metrics_typ = eval_on_typical_holdout(model)
    metrics_full = eval_on_full_holdout(model)
    n_params = int(sum(p.numel() for p in model.parameters()))
    return {
        "kind": "torch",
        "fitted": model,
        "val_mae": float(val_mae),
        "epochs_used": int(epochs_used),
        "n_params": n_params,
        "metrics_typical": metrics_typ,
        "metrics_full": metrics_full,
    }



In [ ]:

# ═══════════════════════════════════════════════════════════════
#  Per-variant DL setup + tuning runner
# ═══════════════════════════════════════════════════════════════
import json as _json

DL_RUN_ARCHS = [
    "FT-Transformer",
    "ExcelFormer",
    "Trompt",
    "BiGRU",
    "GrowNet",
    "SAINT",
    "CNN1D",
    "TabTransformer",
    "AutoInt",
]
# Как в DL_tuning.ipynb (ARCH_TRIALS = {k:50 for k in ARCH_TRIALS_DEFAULT})
ARCH_TRIALS_DEFAULT = {
    "MLP": 40, "SNN": 40, "GatedMLP": 40, "GANDALF": 40, "DCNv2": 40,
    "ResMLP": 30, "CNN1D": 30, "BiGRU": 30, "DAE-MLP": 30, "TabNet": 30,
    "Net-DNF": 30, "ARM-Net": 30, "PTaRL": 30, "SwitchTab": 30, "TabR": 30,
    "FT-Transformer": 25, "TabTransformer": 25, "SAINT": 25, "AutoInt": 25,
    "Trompt": 25, "ExcelFormer": 25, "NODE": 25, "GRANDE": 25,
    "GrowNet": 20,
}
ARCH_TRIALS = {k: 50 for k in ARCH_TRIALS_DEFAULT}
DL_TRIALS_PER_ARCH = 50
DL_INNER_VAL_FRAC = 0.2

def setup_dl_for_variant(Xtr_aug, ytr, Xte_typ_aug, yte_typ, Xte_full_aug, yte_full):
    global X_dl_tr, y_dl_tr, X_dl_va, y_dl_va, X_dl_te, X_full_s, X_te_full_s
    global X_tr_t, y_tr_t, X_va_t, y_va_t, X_te_t, X_te_stress_t
    global X_full_t, y_full_t, X_te_full_t, X_te_fullscale_stress_t
    global INPUT_DIM, sc, sc_full
    global Xtrain, ytrain, Xtest, ytest, ytest_typical, ytest_full

    Xtrain = Xtr_aug.copy()
    ytrain = pd.Series(ytr).reset_index(drop=True)
    Xtest = Xte_typ_aug.copy()
    ytest = pd.Series(yte_typ).reset_index(drop=True)
    ytest_typical = ytest.copy()
    ytest_full = pd.Series(yte_full).reset_index(drop=True)

    val_cut = int(len(Xtrain) * (1 - DL_INNER_VAL_FRAC))
    Xtr_arr = Xtrain.values.astype(np.float32) if hasattr(Xtrain, "values") else np.asarray(Xtrain, dtype=np.float32)
    ytr_arr = ytrain.values.astype(np.float32) if hasattr(ytrain, "values") else np.asarray(ytrain, dtype=np.float32)
    Xte_typ_arr = Xtest.values.astype(np.float32) if hasattr(Xtest, "values") else np.asarray(Xtest, dtype=np.float32)
    Xte_full_arr = Xte_full_aug.values.astype(np.float32) if hasattr(Xte_full_aug, "values") else np.asarray(Xte_full_aug, dtype=np.float32)

    X_dl_tr = Xtr_arr[:val_cut]
    y_dl_tr = ytr_arr[:val_cut]
    X_dl_va = Xtr_arr[val_cut:]
    y_dl_va = ytr_arr[val_cut:]

    sc = StandardScaler()
    X_dl_tr = sc.fit_transform(X_dl_tr)
    X_dl_va = sc.transform(X_dl_va)
    X_dl_te = sc.transform(Xte_typ_arr)
    X_dl_te_full = sc.transform(Xte_full_arr)

    sc_full = StandardScaler()
    X_full_s = sc_full.fit_transform(Xtr_arr)
    X_te_full_s = sc_full.transform(Xte_typ_arr)
    X_te_fullscale_stress = sc_full.transform(Xte_full_arr)

    for arr in [X_dl_tr, X_dl_va, X_dl_te, X_dl_te_full, X_full_s, X_te_full_s, X_te_fullscale_stress]:
        np.nan_to_num(arr, copy=False)

    X_tr_t = torch.from_numpy(X_dl_tr)
    y_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
    X_va_t = torch.from_numpy(X_dl_va)
    y_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
    X_te_t = torch.from_numpy(X_dl_te)
    X_te_stress_t = torch.from_numpy(X_dl_te_full)
    X_full_t = torch.from_numpy(X_full_s)
    y_full_t = torch.from_numpy(ytr_arr).unsqueeze(1)
    X_te_full_t = torch.from_numpy(X_te_full_s)
    X_te_fullscale_stress_t = torch.from_numpy(X_te_fullscale_stress)

    INPUT_DIM = X_dl_tr.shape[1]
    print(f"  setup_dl_for_variant: INPUT_DIM={INPUT_DIM}, train={X_dl_tr.shape}, val={X_dl_va.shape}, test_typ={X_dl_te.shape}, test_full={X_dl_te_full.shape}")

def run_variant_dl_tuning(Xtr_aug, Xte_typ_aug, Xte_full_aug, title="experiment",
                          archs=None, n_trials_per_arch=None):
    if archs is None:
        archs = DL_RUN_ARCHS
    if n_trials_per_arch is None:
        n_trials_per_arch = DL_TRIALS_PER_ARCH

    print("=" * 95)
    print(title)
    print("=" * 95)

    setup_dl_for_variant(
        Xtr_aug, meta_ytr0,
        Xte_typ_aug, meta_yte_typ0,
        Xte_full_aug, meta_yte0,
    )

    rows = []
    for arch_name in archs:
        if arch_name not in OBJECTIVE_MAP:
            print(f"  skip {arch_name}: no objective")
            continue
        arch_n_trials = ARCH_TRIALS.get(arch_name, n_trials_per_arch)
        t0 = time.time()
        seed_everything(SEED)
        sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, warn_independent_sampling=False)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=min(8, max(3, arch_n_trials // 3)), n_warmup_steps=20)
        study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
        if arch_name in BASELINE_TRIAL_PARAMS:
            try:
                study.enqueue_trial(serialize_params(BASELINE_TRIAL_PARAMS[arch_name]))
            except Exception:
                pass
        try:
            study.optimize(OBJECTIVE_MAP[arch_name], n_trials=arch_n_trials, show_progress_bar=False)
        except Exception as e:
            print(f"  {arch_name}: optuna error: {type(e).__name__}: {e}")
            continue
        bp = serialize_params(study.best_params)
        try:
            fit = fit_best_inner_model(arch_name, bp)
        except Exception as e:
            print(f"  {arch_name}: refit error: {type(e).__name__}: {e}")
            continue
        rows.append({
            "model": arch_name,
            "cv_MAE_internal": round(float(study.best_value), 4),
            "refit_val_MAE": round(float(fit["val_mae"]), 4),
            "holdout_typical_MAE": round(float(fit["metrics_typical"]["MAE"]), 4),
            "holdout_full_MAE":    round(float(fit["metrics_full"]["MAE"]), 4),
            "RMSE_typical":        round(float(fit["metrics_typical"]["RMSE"]), 4),
            "RMSE_full":           round(float(fit["metrics_full"]["RMSE"]), 4),
            "MdAE_typical":        round(float(fit["metrics_typical"]["MdAE"]), 4),
            "MdAE_full":           round(float(fit["metrics_full"]["MdAE"]), 4),
            "n_params":            int(fit["n_params"]),
            "epochs_used":         int(fit["epochs_used"]),
            "best_params":         _json.dumps(bp, ensure_ascii=False),
            "time_s":              round(time.time() - t0, 1),
        })
        print(f"  {arch_name:<16s} | val={study.best_value:.3f} | typ={fit['metrics_typical']['MAE']:.3f} | full={fit['metrics_full']['MAE']:.3f} | t={time.time()-t0:.0f}s")
        clear_device_cache()

    out = pd.DataFrame(rows).sort_values(
        ["cv_MAE_internal", "holdout_typical_MAE", "holdout_full_MAE"]
    ).reset_index(drop=True)
    display(out)
    return out


## Run all DL_baseline models on all OFAT configs

In [ ]:

# ═══════════════════════════════════════════════════════════════
#  Run all DL_baseline architectures on all 20 OFAT configs
# ═══════════════════════════════════════════════════════════════
BASELINE_PATH = OUTPUT_DIR / "all_20_ofat_dl_baseline_models.csv"
BEST_SCHEME_PATH = OUTPUT_DIR / "best_ofat_scheme_by_dl_model.csv"
BEST_BY_CONFIG_PATH = OUTPUT_DIR / "best_dl_model_by_ofat_config.csv"

BASELINE_EPOCHS = 300
BASELINE_LR = 1e-3
BASELINE_WD = 1e-4
BASELINE_BS = 256
BASELINE_PATIENCE = 30
SELECTED_DL_MODELS = [
    "ExcelFormer",
    "FT-Transformer",
    "BiGRU",
    "GrowNet",
    "Trompt",
    "TabTransformer",
    "AutoInt",
    "CNN1D",
    "ResMLP",
    "SAINT",
]

ONLY_MODELS = SELECTED_DL_MODELS

BASELINE_MODEL_NAMES = [m for m in SELECTED_DL_MODELS if m in list(DEFAULTS.keys()) + ["GrowNet"]]
if ONLY_MODELS is not None:
    only = set(ONLY_MODELS)
    BASELINE_MODEL_NAMES = [m for m in BASELINE_MODEL_NAMES if m in only]

print("DL baseline models:", len(BASELINE_MODEL_NAMES), BASELINE_MODEL_NAMES)

def run_baseline_arch_current_config(name):
    seed_everything(SEED)
    t0 = time.time()

    if name == "GrowNet":
        stages, val_mae, total_ep = train_grownet(
            n_stages=5, hidden=32, lr=0.01, wd=1e-4,
            step_size=0.1, bs=BASELINE_BS, patience=BASELINE_PATIENCE, dropout=0.1,
        )
        n_params = int(sum(sum(p.numel() for p in s.parameters()) for s in stages))
        m_typ = eval_grownet_on_typical_holdout(stages, step_size=0.1)
        m_full = eval_grownet_on_full_holdout(stages, step_size=0.1)
        epochs_used = int(total_ep)
    else:
        model = DEFAULTS[name]()
        aux_w = AUX_W.get(name, 0.0)
        n_params = int(sum(p.numel() for p in model.parameters()))
        model, val_mae, epochs_used = train_model(
            model,
            epochs=BASELINE_EPOCHS,
            lr=BASELINE_LR,
            wd=BASELINE_WD,
            bs=BASELINE_BS,
            patience=BASELINE_PATIENCE,
            aux_w=aux_w,
        )
        if name == "TabR":
            model.build_store(X_tr_t.to(device), y_tr_t.to(device))
        m_typ = eval_on_typical_holdout(model)
        m_full = eval_on_full_holdout(model)

    elapsed = time.time() - t0
    clear_device_cache()
    return {
        "model": name,
        "model_norm": name,
        "inner_val_MAE": float(val_mae),
        "CV_MAE_mean": float(val_mae),
        "CV_MAE_std": np.nan,
        "CV_MAE_folds": json.dumps([], ensure_ascii=False),
        "holdout_typical_MAE": float(m_typ["MAE"]),
        "holdout_typical_RMSE": float(m_typ["RMSE"]),
        "holdout_typical_MdAE": float(m_typ["MdAE"]),
        "holdout_full_MAE": float(m_full["MAE"]),
        "holdout_full_RMSE": float(m_full["RMSE"]),
        "holdout_full_MdAE": float(m_full["MdAE"]),
        "n_params": int(n_params),
        "epochs_used": int(epochs_used),
        "fit_seconds": round(float(elapsed), 3),
    }

if BASELINE_PATH.exists() and not FORCE_RERUN_BASELINE:
    print("Loading existing DL baseline results:", BASELINE_PATH.resolve())
    baseline_results = pd.read_csv(BASELINE_PATH)
else:
    baseline_rows = []
    expected_runs = len(ofat_configs) * len(BASELINE_MODEL_NAMES)
    run_i = 0

    for cfg_i, cfg in ofat_configs.iterrows():
        print(f"\n[{cfg_i + 1}/{len(ofat_configs)}] {cfg['stage']} :: {cfg['config_slug']}")
        X_train_all, X_test_typical, X_test_full = load_config_matrices(cfg["config_dir"])
        print("  X_train_all:", X_train_all.shape, "X_test_typical:", X_test_typical.shape, "X_test_full:", X_test_full.shape)
        setup_dl_for_variant(X_train_all, y_train_all, X_test_typical, y_test_typical, X_test_full, y_test_full)

        for model_name in BASELINE_MODEL_NAMES:
            run_i += 1
            try:
                row = run_baseline_arch_current_config(model_name)
                row.update({
                    "provider": PROVIDER,
                    "stage": cfg["stage"],
                    "stage_num": int(cfg["stage_num"]),
                    "config_slug": cfg["config_slug"],
                    "config_dir": str(cfg["config_dir"]),
                })
                baseline_rows.append(row)
                print(f"  [{run_i:03d}/{expected_runs}] {model_name:<18s} val MAE={row['inner_val_MAE']:.2f} typ={row['holdout_typical_MAE']:.2f} full={row['holdout_full_MAE']:.2f}")
            except Exception as e:
                print(f"  [{run_i:03d}/{expected_runs}] {model_name:<18s} Ошибка: {type(e).__name__}: {e}")
                clear_device_cache()

        pd.DataFrame(baseline_rows).to_csv(BASELINE_PATH, index=False)

    baseline_results = pd.DataFrame(baseline_rows)
    baseline_results = baseline_results.sort_values(["inner_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"]).reset_index(drop=True)
    baseline_results.to_csv(BASELINE_PATH, index=False)

best_scheme_by_model = (
    baseline_results.sort_values(["inner_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"])
    .groupby("model_norm", as_index=False)
    .head(1)
    .sort_values(["inner_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"])
    .reset_index(drop=True)
)
best_by_config = (
    baseline_results.sort_values(["inner_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"])
    .groupby(["stage", "config_slug"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_scheme_by_model.to_csv(BEST_SCHEME_PATH, index=False)
best_by_config.to_csv(BEST_BY_CONFIG_PATH, index=False)

print("Best OFAT scheme by DL model:")
display(best_scheme_by_model)
print("Best DL model by OFAT config:")
display(best_by_config)


## Tune each DL model on its best OFAT scheme with DL_tuning search spaces

In [ ]:

# ═══════════════════════════════════════════════════════════════
#  Tune each DL model on its best OFAT scheme with DL_tuning search spaces
# ═══════════════════════════════════════════════════════════════
TUNING_PATH = OUTPUT_DIR / "best_scheme_dl_tuning_results.csv"
TRIAL_DUMP_DIR = OUTPUT_DIR / "optuna_trials_by_model"
TRIAL_DUMP_DIR.mkdir(parents=True, exist_ok=True)

# Same schedule as the DL_tuning-derived notebooks: ARCH_TRIALS = {k: 50 for k in ARCH_TRIALS_DEFAULT}.
ARCH_TRIALS_DEFAULT = {
    "MLP": 40, "SNN": 40, "GatedMLP": 40, "GANDALF": 40, "DCNv2": 40,
    "ResMLP": 30, "CNN1D": 30, "BiGRU": 30, "DAE-MLP": 30, "TabNet": 30,
    "Net-DNF": 30, "ARM-Net": 30, "PTaRL": 30, "SwitchTab": 30, "TabR": 30,
    "FT-Transformer": 25, "TabTransformer": 25, "SAINT": 25, "AutoInt": 25,
    "Trompt": 25, "ExcelFormer": 25, "NODE": 25, "GRANDE": 25,
    "GrowNet": 20,
}
ARCH_TRIALS = {k: 50 for k in ARCH_TRIALS_DEFAULT}
TUNE_ONLY_MODELS = SELECTED_DL_MODELS
ENQUEUE_BASELINE_TRIAL = True

if TUNING_PATH.exists() and not FORCE_RERUN_TUNING:
    print("Loading existing DL tuning results:", TUNING_PATH.resolve())
    tuning_results = pd.read_csv(TUNING_PATH)
else:
    tuning_rows = []
    candidates = best_scheme_by_model.sort_values(["inner_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"]).copy()
    if TUNE_ONLY_MODELS is not None:
        candidates = candidates[candidates["model_norm"].isin(TUNE_ONLY_MODELS)].copy()

    for _, best_row in candidates.iterrows():
        arch_name = best_row["model_norm"]
        if arch_name not in OBJECTIVE_MAP:
            print(f"\nSkipping {arch_name}: no objective in DL_tuning.ipynb")
            continue

        print(f"\n===== DL tuning {arch_name} on best OFAT scheme: {best_row['stage']} :: {best_row['config_slug']} =====")
        config_dir = Path(best_row["config_dir"])
        X_train_all, X_test_typical, X_test_full = load_config_matrices(config_dir)
        setup_dl_for_variant(X_train_all, y_train_all, X_test_typical, y_test_typical, X_test_full, y_test_full)

        arch_n_trials = int(ARCH_TRIALS.get(arch_name, 50))
        t0 = time.time()
        seed_everything(SEED)
        sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, warn_independent_sampling=False)
        pruner = optuna.pruners.MedianPruner(n_startup_trials=min(8, max(3, arch_n_trials // 3)), n_warmup_steps=20)
        study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
        if ENQUEUE_BASELINE_TRIAL and arch_name in BASELINE_TRIAL_PARAMS:
            try:
                study.enqueue_trial(serialize_params(BASELINE_TRIAL_PARAMS[arch_name]))
            except Exception:
                pass

        try:
            study.optimize(OBJECTIVE_MAP[arch_name], n_trials=arch_n_trials, show_progress_bar=False)
            bp = serialize_params(study.best_params)
            fit = fit_best_inner_model(arch_name, bp)
        except Exception as e:
            print(f"  {arch_name}: tuning error: {type(e).__name__}: {e}")
            clear_device_cache()
            continue

        trials_df = study.trials_dataframe()
        trials_df.to_csv(TRIAL_DUMP_DIR / f"optuna_trials_best_scheme_{re.sub(r'[^a-zA-Z0-9._-]+', '_', arch_name)}.csv", index=False)

        row = {
            "provider": PROVIDER,
            "model_norm": arch_name,
            "source_baseline_model": best_row["model"],
            "stage": best_row["stage"],
            "stage_num": int(best_row["stage_num"]),
            "config_slug": best_row["config_slug"],
            "config_dir": str(config_dir),
            "baseline_inner_val_MAE": float(best_row["inner_val_MAE"]),
            "baseline_holdout_typical_MAE": float(best_row["holdout_typical_MAE"]),
            "baseline_holdout_full_MAE": float(best_row["holdout_full_MAE"]),
            "optuna_best_val_MAE": float(study.best_value),
            "refit_val_MAE": float(fit["val_mae"]),
            "holdout_typical_MAE": float(fit["metrics_typical"]["MAE"]),
            "holdout_typical_RMSE": float(fit["metrics_typical"]["RMSE"]),
            "holdout_typical_MdAE": float(fit["metrics_typical"]["MdAE"]),
            "holdout_full_MAE": float(fit["metrics_full"]["MAE"]),
            "holdout_full_RMSE": float(fit["metrics_full"]["RMSE"]),
            "holdout_full_MdAE": float(fit["metrics_full"]["MdAE"]),
            "n_trials": arch_n_trials,
            "n_params": int(fit["n_params"]),
            "epochs_used": int(fit["epochs_used"]),
            "best_params": json.dumps(bp, ensure_ascii=False),
            "tune_seconds": round(time.time() - t0, 3),
        }
        tuning_rows.append(row)
        pd.DataFrame(tuning_rows).to_csv(TUNING_PATH, index=False)
        print(f"  {arch_name:<16s} val={row['optuna_best_val_MAE']:.3f} typ={row['holdout_typical_MAE']:.3f} full={row['holdout_full_MAE']:.3f}")
        clear_device_cache()

    tuning_results = pd.DataFrame(tuning_rows)
    if len(tuning_results):
        tuning_results = tuning_results.sort_values(["optuna_best_val_MAE", "holdout_typical_MAE", "holdout_full_MAE"]).reset_index(drop=True)
        tuning_results.to_csv(TUNING_PATH, index=False)

print("DL tuning results:")
display(tuning_results)


## Final report

In [ ]:

# Final report
plot_top = min(30, len(baseline_results))
if plot_top:
    top_df = baseline_results.head(plot_top).copy()
    top_df["label"] = top_df["model"] + " | " + top_df["stage"].str.replace("stage_", "s", regex=False)
    plt.figure(figsize=(12, 8), dpi=130)
    sns.barplot(data=top_df, x="inner_val_MAE", y="label", hue="stage", dodge=False)
    plt.title(f"{PROVIDER}: top DL baseline candidates across OFAT configs")
    plt.xlabel("Inner validation MAE")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

if len(tuning_results):
    tune_plot = tuning_results.copy()
    tune_plot["label"] = tune_plot["model_norm"] + " | " + tune_plot["stage"].str.replace("stage_", "s", regex=False)
    plt.figure(figsize=(10, max(5, 0.35 * len(tune_plot))), dpi=130)
    sns.barplot(data=tune_plot, x="optuna_best_val_MAE", y="label", color="#4c78a8")
    plt.title(f"{PROVIDER}: DL_tuning search space on each model's best OFAT scheme")
    plt.xlabel("Optuna best inner validation MAE")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

report = {
    "provider": PROVIDER,
    "data_path": str(DATA_PATH),
    "artifact_root": str(ARTIFACT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "n_ofat_configs": int(len(ofat_configs)),
    "selected_dl_models": SELECTED_DL_MODELS,
    "n_baseline_models": int(len(BASELINE_MODEL_NAMES)),
    "n_expected_baseline_runs": int(len(ofat_configs) * len(BASELINE_MODEL_NAMES)),
    "n_baseline_rows": int(len(baseline_results)),
    "n_models_with_best_scheme": int(len(best_scheme_by_model)),
    "n_tuned_models": int(len(tuning_results)),
    "dl_baseline_source": "DL_baseline.ipynb defaults + GrowNet",
    "dl_tuning_source": "DL_tuning.ipynb Optuna objectives/search spaces",
    "arch_trials": ARCH_TRIALS,
    "best_baseline_overall": baseline_results.head(1).to_dict(orient="records"),
    "best_tuned_overall": tuning_results.head(1).to_dict(orient="records") if len(tuning_results) else [],
}

with open(OUTPUT_DIR / "run_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print(json.dumps(report, ensure_ascii=False, indent=2, default=str))
